# Orthosync · Entrenamiento de los modelos (Kaggle, GPU gratuita)
**Proyecto de grado de Juan Miguel Huanca Uruña** · Sistema de apoyo al diagnóstico de tumores óseos

Este cuaderno entrena los cinco modelos con el dataset BTXRD y los deja listos para el servidor web:

1. Normal vs. lesión (ResNet18)
2. Benigno vs. maligno (ResNet18)
3. Hueso de la radiografía (ResNet18, para el visor 3D)
4. Aspecto de la lesión: crece hacia afuera, lítica o mixta (ResNet18, decide cómo se dibuja en 3D)
5. Localización de la lesión (YOLO11s a 1024 px)

### Casos corregidos por el médico (opcional)
Si exportaste casos desde Orthosync (*Validación → Exportar casos corregidos*), súbelos a Kaggle como un dataset
(*Add Input → Upload → New Dataset*, sube el ZIP tal cual) y agrégalo a este cuaderno. Se suman solo al entrenamiento;
la prueba sigue siendo BTXRD para que los resultados se puedan comparar.

### Antes de ejecutar (panel derecho de Kaggle → *Settings*)
- **Accelerator:** GPU T4 x2 (o P100)
- **Internet:** On (Kaggle pide verificar tu teléfono la primera vez)

### Cómo ejecutarlo
Botón **Save Version → Save & Run All (Commit)**. Así corre en segundo plano aunque cierres la pestaña.
Al terminar, abre la versión guardada → pestaña **Output** → descarga **orthosync_modelos.zip**.

In [ ]:
import os, sys, subprocess, torch
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
IA = os.path.join(BASE, 'ia'); os.makedirs(IA, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'onnx', 'onnxruntime', 'openpyxl'])
print('GPU disponible:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU: activa el acelerador en Settings')

## 1. Código del entrenamiento
Cada celda guarda un archivo del proyecto. No hace falta modificarlas.

In [ ]:
%%writefile {IA}/nucleo.py
"""Núcleo del sistema: preprocesamiento, modelos, Grad-CAM y análisis completo
de una radiografía. Lo usan los scripts de entrenamiento, la evaluación y la app."""
import json
import os

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from torchvision import models, transforms

RAIZ = os.path.dirname(os.path.abspath(__file__))
DIR_MODELOS = os.path.join(RAIZ, "modelos")
DIR_RESULTADOS = os.path.join(RAIZ, "resultados")
DIR_PREPARADOS = os.path.join(RAIZ, "datos_preparados")
RUTA_DIVISION = os.path.join(DIR_PREPARADOS, "division.csv")
RUTA_CONFIG = os.path.join(DIR_PREPARADOS, "config.json")

MODELO_TUMOR = os.path.join(DIR_MODELOS, "clasificador_tumor.pt")
MODELO_MALIGNIDAD = os.path.join(DIR_MODELOS, "clasificador_malignidad.pt")
MODELO_DETECTOR = os.path.join(DIR_MODELOS, "detector.pt")
MODELO_HUESO = os.path.join(DIR_MODELOS, "clasificador_hueso.pt")
MODELO_ASPECTO = os.path.join(DIR_MODELOS, "clasificador_aspecto.pt")

# Aspecto radiológico de la lesión según el tipo de tumor de BTXRD (columna -> aspecto). Se usa para dibujar el 3D:
#   crecimiento: la lesión sobresale del hueso (osteocondromas)
#   litica:      la lesión destruye hueso y se ve como un hueco (quiste óseo, tumor de células gigantes, osteofibroma)
#   mixta:       destruye hueso y forma masa (osteosarcoma)
# "other bt", "other mt" y el osteocondroma sinovial no se usan porque su aspecto es variable.
ASPECTOS = {"crecimiento": ["osteochondroma", "multiple osteochondromas"],
            "litica": ["simple bone cyst", "giant cell tumor", "osteofibroma"],
            "mixta": ["osteosarcoma"]}

# clave interna -> nombre de la columna en BTXRD
HUESOS = {"mano": "hand", "cubito": "ulna", "radio": "radius", "humero": "humerus", "pie": "foot",
          "tibia": "tibia", "perone": "fibula", "femur": "femur", "cadera": "hip bone"}

MEDIA, DESV = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


# --------------------------------------------------------------------------
# Utilidades de archivos
# --------------------------------------------------------------------------
def buscar(raiz, nombre, carpeta=False):
    """Busca un archivo o carpeta por nombre dentro de raiz (sin importar mayúsculas)."""
    for d, carpetas, archivos in os.walk(raiz):
        for it in (carpetas if carpeta else archivos):
            if it.lower() == nombre.lower():
                return os.path.join(d, it)
    return None


def columna(df, nombre):
    limpio = lambda s: str(s).lower().replace(" ", "").replace("_", "").replace("-", "")
    for c in df.columns:
        if limpio(c) == limpio(nombre):
            return c
    return None


def dispositivo():
    return "cuda" if torch.cuda.is_available() else "cpu"


# --------------------------------------------------------------------------
# Lectura y preprocesamiento de imágenes
# --------------------------------------------------------------------------
def leer_imagen(origen, nombre=None) -> Image.Image:
    """Lee JPG/PNG o DICOM (.dcm) y devuelve una imagen en escala de grises.
    De un DICOM solo se usan los píxeles: no se lee ni se guarda ningún dato del paciente."""
    nombre = (nombre or (origen if isinstance(origen, str) else getattr(origen, "name", ""))).lower()
    if nombre.endswith((".dcm", ".dicom")):
        import pydicom
        ds = pydicom.dcmread(origen)
        a = ds.pixel_array.astype(np.float32)
        if a.ndim == 3:  # varias tramas o color: nos quedamos con un plano
            a = a[..., 0] if a.shape[-1] in (3, 4) else a[0]
        a = (a - a.min()) / max(float(a.max() - a.min()), 1e-6) * 255.0
        if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
            a = 255.0 - a
        return Image.fromarray(a.astype(np.uint8), mode="L")
    return Image.open(origen).convert("L")


def cuadrar(img: Image.Image, tam: int) -> Image.Image:
    """Escala de grises -> mejora de contraste (CLAHE) -> redimensionar con relleno."""
    gris = np.asarray(img.convert("L"))
    gris = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gris)
    img = Image.fromarray(gris)
    w, h = img.size
    escala = tam / max(w, h)
    img = img.resize((max(1, round(w * escala)), max(1, round(h * escala))), Image.BILINEAR)
    lienzo = Image.new("L", (tam, tam), 0)
    lienzo.paste(img, ((tam - img.width) // 2, (tam - img.height) // 2))
    return lienzo.convert("RGB")


def transformacion(entrenamiento: bool):
    pasos = []
    if entrenamiento:  # aumento de datos
        pasos += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomAffine(degrees=12, translate=(0.05, 0.05), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    pasos += [transforms.ToTensor(), transforms.Normalize(MEDIA, DESV)]
    return transforms.Compose(pasos)


def asegurar_cache(rutas, tam):
    """Guarda una copia ya preprocesada de cada imagen (acelera mucho el entrenamiento)."""
    carpeta = os.path.join(DIR_PREPARADOS, f"cache_{tam}")
    os.makedirs(carpeta, exist_ok=True)
    salida = []
    for i, r in enumerate(rutas):
        destino = os.path.join(carpeta, os.path.splitext(os.path.basename(r))[0] + ".png")
        if not os.path.exists(destino):
            cuadrar(leer_imagen(r), tam).save(destino)
        salida.append(destino)
        if (i + 1) % 500 == 0:
            print(f"  preparando imágenes {i + 1}/{len(rutas)}", flush=True)
    return salida


# --------------------------------------------------------------------------
# Clasificadores (ResNet18 con transfer learning)
# --------------------------------------------------------------------------
def crear_modelo(n_clases=2, preentrenado=True) -> nn.Module:
    red = None
    if preentrenado:
        try:
            red = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        except Exception as e:
            print(f"AVISO: no se pudieron descargar los pesos preentrenados ({e}).\n"
                  "       Se entrena desde cero; los resultados serán peores. Revisa tu internet.")
    if red is None:
        red = models.resnet18(weights=None)
    red.fc = nn.Linear(red.fc.in_features, n_clases)
    return red


def cargar_clasificador(ruta):
    datos = torch.load(ruta, map_location="cpu", weights_only=False)
    red = crear_modelo(len(datos["clases"]), preentrenado=False)
    red.load_state_dict(datos["pesos"])
    red.eval()
    return red, datos


def predecir_con_gradcam(red, x):
    """Devuelve (probabilidad de la clase positiva, mapa Grad-CAM normalizado 0-1)."""
    guardado = {}
    h1 = red.layer4.register_forward_hook(lambda m, i, o: guardado.__setitem__("act", o))
    h2 = red.layer4.register_full_backward_hook(lambda m, gi, go: guardado.__setitem__("grad", go[0]))
    try:
        red.zero_grad()
        salida = red(x)
        prob = float(F.softmax(salida, dim=1)[0, 1].detach())
        salida[0, 1].backward()
    finally:
        h1.remove(); h2.remove()
    pesos = guardado["grad"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((pesos * guardado["act"]).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = cam.detach().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return prob, cam


def superponer_calor(base: Image.Image, cam: np.ndarray) -> Image.Image:
    gris = np.asarray(base.convert("RGB")).astype(np.float32) / 255.0
    calor = np.stack([np.ones_like(cam), cam ** 2, np.zeros_like(cam)], axis=-1)
    alfa = (cam ** 1.5 * 0.6)[..., None]
    return Image.fromarray(((gris * (1 - alfa) + calor * alfa) * 255).astype(np.uint8))


# --------------------------------------------------------------------------
# Sistema completo: las tres etapas sobre una radiografía
# --------------------------------------------------------------------------
class Sistema:
    """Etapa 1: ¿normal o lesión?   Etapa 2: ¿dónde?   Etapa 3: ¿benigna o maligna?"""

    def __init__(self):
        if not os.path.exists(MODELO_TUMOR):
            raise FileNotFoundError("Falta modelos/clasificador_tumor.pt. Entrena primero el sistema.")
        self.red_tumor, self.info_tumor = cargar_clasificador(MODELO_TUMOR)
        self.red_mal = self.info_mal = self.detector = None
        if os.path.exists(MODELO_MALIGNIDAD):
            self.red_mal, self.info_mal = cargar_clasificador(MODELO_MALIGNIDAD)
        if os.path.exists(MODELO_DETECTOR):
            from ultralytics import YOLO
            self.detector = YOLO(MODELO_DETECTOR)

    @property
    def umbral_tumor(self):
        return float(self.info_tumor.get("umbral", 0.5))

    @property
    def umbral_malignidad(self):
        return float(self.info_mal.get("umbral", 0.5)) if self.info_mal else 0.5

    def probabilidades(self, imagen: Image.Image):
        """Solo las dos probabilidades (rápido; lo usa la evaluación)."""
        with torch.no_grad():
            x = transformacion(False)(cuadrar(imagen, self.info_tumor["tam"])).unsqueeze(0)
            p_tumor = float(F.softmax(self.red_tumor(x), 1)[0, 1])
            p_mal = None
            if self.red_mal is not None:
                x = transformacion(False)(cuadrar(imagen, self.info_mal["tam"])).unsqueeze(0)
                p_mal = float(F.softmax(self.red_mal(x), 1)[0, 1])
        return p_tumor, p_mal

    def detectar(self, imagen: Image.Image, confianza=0.25):
        if self.detector is None:
            return []
        r = self.detector.predict(imagen.convert("RGB"), conf=confianza, verbose=False)[0]
        return [(*[float(v) for v in caja], float(c))
                for caja, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy())]

    def analizar(self, imagen: Image.Image, umbral_tumor=None, umbral_malignidad=None, confianza_cajas=0.25):
        umbral_tumor = self.umbral_tumor if umbral_tumor is None else umbral_tumor
        umbral_malignidad = self.umbral_malignidad if umbral_malignidad is None else umbral_malignidad
        imagen = imagen.convert("L")

        # Etapa 1: clasificación + mapa de calor
        base = cuadrar(imagen, self.info_tumor["tam"])
        p_tumor, cam = predecir_con_gradcam(self.red_tumor, transformacion(False)(base).unsqueeze(0))
        sospechoso = p_tumor >= umbral_tumor

        # Etapa 2: localización
        cajas = self.detectar(imagen, confianza_cajas)
        con_cajas = imagen.convert("RGB")
        dibujo = ImageDraw.Draw(con_cajas)
        grosor = max(2, round(max(con_cajas.size) / 300))
        tam_letra = max(12, round(max(con_cajas.size) / 40))
        try:
            letra = ImageFont.load_default(size=tam_letra)
        except TypeError:  # versiones antiguas de Pillow
            letra = ImageFont.load_default()
        for x1, y1, x2, y2, c in cajas:
            dibujo.rectangle([x1, y1, x2, y2], outline=(0, 255, 0), width=grosor)
            dibujo.text((x1 + grosor, max(0, y1 - tam_letra - 2)), f"{c:.0%}", fill=(0, 255, 0), font=letra)

        # Etapa 3: benigno vs. maligno (solo si hay sospecha)
        p_mal = None
        if sospechoso and self.red_mal is not None:
            with torch.no_grad():
                x = transformacion(False)(cuadrar(imagen, self.info_mal["tam"])).unsqueeze(0)
                p_mal = float(F.softmax(self.red_mal(x), 1)[0, 1])

        if not sospechoso:
            conclusion = "Sin hallazgos sospechosos"
        elif p_mal is None:
            conclusion = "Lesión sospechosa"
        elif p_mal >= umbral_malignidad:
            conclusion = "Lesión sospechosa con características de malignidad"
        else:
            conclusion = "Lesión sospechosa con características de benignidad"

        return {
            "p_tumor": p_tumor, "umbral_tumor": umbral_tumor, "sospechoso": bool(sospechoso),
            "p_maligno": p_mal, "umbral_malignidad": umbral_malignidad,
            "cajas": cajas, "discordancia": bool(cajas) != bool(sospechoso) and self.detector is not None,
            "conclusion": conclusion,
            "img_original": imagen.convert("RGB"), "img_cajas": con_cajas,
            "img_calor": superponer_calor(base, cam),
        }


def leer_json(ruta, defecto=None):
    if os.path.exists(ruta):
        with open(ruta, encoding="utf-8") as f:
            return json.load(f)
    return defecto


In [ ]:
%%writefile {IA}/preparar_datos.py
"""Paso 1: lee BTXRD, aplica el alcance del proyecto y divide en
entrenamiento / validación / prueba (la misma división para todos los modelos).

Uso:  python preparar_datos.py --datos "C:\\tesis\\BTXRD"
      python preparar_datos.py --datos ... --extra casos_clinica   (suma los casos corregidos exportados desde Orthosync)
"""
import argparse, glob, json, os, shutil

import pandas as pd
from sklearn.model_selection import train_test_split

from nucleo import ASPECTOS, DIR_PREPARADOS, HUESOS, RAIZ, RUTA_CONFIG, RUTA_DIVISION, buscar, columna


def casos_clinica(carpetas):
    """Casos valorados por el traumatólogo y exportados desde Orthosync (casos.csv + images/ + labels/).
    Se usan SOLO para entrenar: la prueba sigue siendo BTXRD, para que los resultados sean comparables."""
    filas = []
    for raiz in carpetas:
        for csv in glob.glob(os.path.join(raiz, "**", "casos.csv"), recursive=True):
            base = os.path.dirname(csv)
            for _, f in pd.read_csv(csv).iterrows():
                ruta = os.path.join(base, "images", f["imagen"])
                if not os.path.exists(ruta):
                    continue
                maligno = str(f.get("maligno", "")).strip()
                clase = "normal" if int(f["tumor"]) == 0 else ("maligno" if maligno in ("1", "1.0") else "benigno")
                lab = os.path.join(base, "labels", os.path.splitext(f["imagen"])[0] + ".txt")
                fila = {"imagen": "clinica_" + f["imagen"], "ruta": ruta, "clase": clase, "conjunto": "entrenamiento",
                        "etiquetas_yolo": lab if os.path.exists(lab) else "", "origen": "clinica"}
                h = str(f.get("hueso", "") or "")
                for clave in HUESOS:
                    fila["hueso_" + clave] = int(h == clave)
                filas.append(fila)
    if filas:
        print(f"Se suman {len(filas)} casos corregidos por el médico (solo para entrenamiento)")
    return pd.DataFrame(filas)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--datos", required=True, help="Carpeta donde descomprimiste BTXRD")
    ap.add_argument("--max-imagenes", type=int, default=0, help="Usar solo N imágenes (prueba rápida)")
    ap.add_argument("--incluir-pelvis", action="store_true")
    ap.add_argument("--extra", nargs="*", default=[], help="Carpetas con casos exportados desde Orthosync")
    a = ap.parse_args()

    tabla = buscar(a.datos, "dataset.xlsx") or buscar(a.datos, "dataset.csv")
    dir_img = buscar(a.datos, "images", carpeta=True)
    dir_anot = buscar(a.datos, "Annotations", carpeta=True)
    if not tabla or not dir_img:
        raise SystemExit(f"No encuentro dataset.xlsx/csv o la carpeta images dentro de {a.datos}")
    df = pd.read_excel(tabla) if tabla.endswith(".xlsx") else pd.read_csv(tabla)

    c_img = columna(df, "image_id") or df.columns[0]
    c_tumor, c_ben, c_mal = columna(df, "tumor"), columna(df, "benign"), columna(df, "malignant")
    if c_tumor is None:
        raise SystemExit(f"No hay columna 'tumor'. Columnas: {list(df.columns)}")

    c_pelvis = columna(df, "pelvis")
    if c_pelvis and not a.incluir_pelvis:
        antes = len(df); df = df[df[c_pelvis] != 1]
        print(f"Se excluyen {antes - len(df)} imágenes de pelvis (fuera del alcance)")

    def clase(f):
        if c_mal and f[c_mal] == 1: return "maligno"
        if c_ben and f[c_ben] == 1: return "benigno"
        if f[c_tumor] == 1: return "tumor_sin_tipo"
        return "normal"

    df = df.copy()
    df["imagen"] = df[c_img].astype(str)
    df["ruta"] = df["imagen"].map(lambda n: os.path.join(dir_img, n))
    df = df[df["ruta"].map(os.path.exists)]
    df["clase"] = df.apply(clase, axis=1)
    if len(df) == 0:
        raise SystemExit("No se encontró ninguna imagen de la tabla dentro de la carpeta images.")

    if a.max_imagenes:
        n = max(a.max_imagenes // df["clase"].nunique(), 6)
        df = pd.concat([g.sample(min(len(g), n), random_state=0) for _, g in df.groupby("clase")])

    # 70 % entrenamiento / 15 % validación / 15 % prueba, manteniendo la proporción de clases
    entr, resto = train_test_split(df, test_size=0.30, stratify=df["clase"], random_state=0)
    val, prueba = train_test_split(resto, test_size=0.50, stratify=resto["clase"], random_state=0)
    df["conjunto"] = "entrenamiento"
    df.loc[val.index, "conjunto"] = "validacion"
    df.loc[prueba.index, "conjunto"] = "prueba"

    # aspecto radiológico de la lesión según el tipo de tumor (para el visor 3D)
    df["aspecto"] = ""
    for aspecto, tipos in ASPECTOS.items():
        for t in tipos:
            c = columna(df, t)
            if c:
                df.loc[pd.to_numeric(df[c], errors="coerce").fillna(0) == 1, "aspecto"] = aspecto
    df.loc[df["clase"] == "normal", "aspecto"] = ""
    df["origen"] = "btxrd"; df["etiquetas_yolo"] = ""

    extras = ["aspecto", "origen", "etiquetas_yolo"] + [c for c in (columna(df, "age"), columna(df, "gender"), columna(df, "center")) if c]
    # hueso(s) que aparecen en cada radiografía (para el clasificador de hueso y el visor 3D)
    for clave, nombre in HUESOS.items():
        c = columna(df, nombre)
        if c:
            df["hueso_" + clave] = (pd.to_numeric(df[c], errors="coerce").fillna(0) == 1).astype(int)
            extras.append("hueso_" + clave)
    os.makedirs(DIR_PREPARADOS, exist_ok=True)
    salida = df[["imagen", "ruta", "clase", "conjunto"] + extras]
    clinica = casos_clinica(a.extra)
    if len(clinica):
        salida = pd.concat([salida, clinica], ignore_index=True)
        for c in salida.columns:
            if c.startswith("hueso_"):
                salida[c] = salida[c].fillna(0).astype(int)
        salida["aspecto"] = salida["aspecto"].fillna("")
    salida.to_csv(RUTA_DIVISION, index=False)
    with open(RUTA_CONFIG, "w", encoding="utf-8") as f:
        json.dump({"datos": os.path.abspath(a.datos), "dir_img": dir_img, "dir_anot": dir_anot}, f, indent=2)

    # imágenes de PRUEBA (los modelos nunca las ven) para probar la aplicación
    ejemplos = os.path.join(RAIZ, "ejemplos_prueba")
    shutil.rmtree(ejemplos, ignore_errors=True); os.makedirs(ejemplos)
    pr = df[df.conjunto == "prueba"]   # solo BTXRD
    for nombre, g in pr.groupby("clase"):
        for r in g["ruta"].sample(min(8, len(g)), random_state=0):
            shutil.copy(r, os.path.join(ejemplos, f"{nombre}_{os.path.basename(r)}"))

    print("\nImágenes por clase y conjunto:")
    print(pd.crosstab(salida["clase"], salida["conjunto"], margins=True).to_string())
    if (salida["aspecto"] != "").any():
        print("\nAspecto de la lesión (para el 3D):", salida[salida["aspecto"] != ""]["aspecto"].value_counts().to_dict())
    print(f"\nDivisión guardada en {RUTA_DIVISION}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {IA}/entrenar_clasificador.py
"""Paso 2 y 3: entrena un clasificador ResNet18.

    python entrenar_clasificador.py --tarea tumor        (normal vs. lesión)
    python entrenar_clasificador.py --tarea malignidad   (benigno vs. maligno)
"""
import argparse, json, os, random, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset

from nucleo import (DIR_MODELOS, DIR_RESULTADOS, MODELO_MALIGNIDAD, MODELO_TUMOR, RUTA_DIVISION,
                    asegurar_cache, crear_modelo, dispositivo, transformacion)

TAREAS = {
    "tumor": {"clases": ["normal", "lesion"], "salida": MODELO_TUMOR},
    "malignidad": {"clases": ["benigno", "maligno"], "salida": MODELO_MALIGNIDAD},
}


class Radiografias(Dataset):
    def __init__(self, rutas, etiquetas, entrenamiento):
        self.rutas, self.etiquetas = list(rutas), [int(e) for e in etiquetas]
        self.t = transformacion(entrenamiento)

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, i):
        return self.t(Image.open(self.rutas[i]).convert("RGB")), self.etiquetas[i]


def metricas(reales, probs, umbral=None):
    reales, probs = np.asarray(reales), np.asarray(probs)
    if umbral is None:  # umbral que mejor separa las clases (índice de Youden)
        fpr, tpr, umbrales = roc_curve(reales, probs)
        umbral = float(np.clip(umbrales[np.argmax(tpr - fpr)], 0.01, 0.99))
    vn, fp, fn, vp = confusion_matrix(reales, probs >= umbral, labels=[0, 1]).ravel()
    return {
        "umbral": umbral,
        "auc": float(roc_auc_score(reales, probs)) if len(set(reales)) > 1 else float("nan"),
        "sensibilidad": float(vp / max(vp + fn, 1)),
        "especificidad": float(vn / max(vn + fp, 1)),
        "exactitud": float((vp + vn) / max(len(reales), 1)),
        "matriz": {"VN": int(vn), "FP": int(fp), "FN": int(fn), "VP": int(vp)},
    }


def predecir(red, cargador, disp):
    red.eval(); probs, reales = [], []
    with torch.no_grad():
        for x, y in cargador:
            probs += list(torch.softmax(red(x.to(disp)), 1)[:, 1].cpu().numpy())
            reales += list(y.numpy())
    return np.array(reales), np.array(probs)


def tabla_de_tarea(tarea):
    if not os.path.exists(RUTA_DIVISION):
        raise SystemExit("Falta la división de datos. Ejecuta primero preparar_datos.py")
    df = pd.read_csv(RUTA_DIVISION)
    if tarea == "tumor":
        df["y"] = (df["clase"] != "normal").astype(int)
    else:
        df = df[df["clase"].isin(["benigno", "maligno"])].copy()
        df["y"] = (df["clase"] == "maligno").astype(int)
    return df.reset_index(drop=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tarea", choices=list(TAREAS), required=True)
    ap.add_argument("--epocas", type=int, default=0)
    ap.add_argument("--tam", type=int, default=0)
    ap.add_argument("--lote", type=int, default=32)
    ap.add_argument("--sin-preentrenar", action="store_true")
    a = ap.parse_args()

    disp = dispositivo()
    gpu = disp == "cuda"
    epocas = a.epocas or ({"tumor": 15, "malignidad": 20} if gpu else {"tumor": 6, "malignidad": 10})[a.tarea]
    tam = a.tam or (384 if gpu else 224)
    random.seed(0); np.random.seed(0); torch.manual_seed(0)
    print(f"=== Clasificador '{a.tarea}' | dispositivo={disp} | épocas={epocas} | imagen={tam}px ===")

    df = tabla_de_tarea(a.tarea)
    df["cache"] = asegurar_cache(df["ruta"], tam)
    entr, val, prueba = (df[df.conjunto == c] for c in ("entrenamiento", "validacion", "prueba"))
    clases = TAREAS[a.tarea]["clases"]
    for nombre, d in (("Entrenamiento", entr), ("Validación", val), ("Prueba", prueba)):
        print(f"{nombre:14s}: {len(d):5d}  ({clases[0]}={sum(d.y == 0)}, {clases[1]}={sum(d.y == 1)})")
    if entr.y.nunique() < 2 or val.y.nunique() < 2:
        raise SystemExit("No hay imágenes de las dos clases; no se puede entrenar esta tarea.")

    mk = lambda d, e: DataLoader(Radiografias(d.cache, d.y, e), batch_size=a.lote, shuffle=e, num_workers=0)
    c_entr, c_val, c_prueba = mk(entr, True), mk(val, False), mk(prueba, False)

    red = crear_modelo(2, preentrenado=not a.sin_preentrenar).to(disp)
    conteo = np.bincount(entr.y, minlength=2)
    pesos = torch.tensor(len(entr) / (2 * conteo), dtype=torch.float32, device=disp)  # contra el desbalance
    perdida = nn.CrossEntropyLoss(weight=pesos)
    opt = torch.optim.AdamW(red.parameters(), lr=3e-4, weight_decay=1e-4)
    plan = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=epocas * len(c_entr))

    os.makedirs(DIR_MODELOS, exist_ok=True); os.makedirs(DIR_RESULTADOS, exist_ok=True)
    salida = TAREAS[a.tarea]["salida"]
    mejor, historial = -1, []
    for ep in range(1, epocas + 1):
        red.train(); t0 = time.time(); suma = 0
        for i, (x, y) in enumerate(c_entr):
            x, y = x.to(disp), y.to(disp)
            opt.zero_grad()
            l = perdida(red(x), y)
            l.backward(); opt.step(); plan.step()
            suma += l.item()
            if (i + 1) % 10 == 0:
                print(f"\r  época {ep}: lote {i + 1}/{len(c_entr)}", end="", flush=True)
        m = metricas(*predecir(red, c_val, disp))
        historial.append({"epoca": ep, "perdida": suma / len(c_entr), **m})
        print(f"\rÉpoca {ep}/{epocas}  pérdida={suma / len(c_entr):.3f}  AUC val={m['auc']:.3f}  "
              f"sens={m['sensibilidad']:.2f}  espec={m['especificidad']:.2f}  ({time.time() - t0:.0f}s)", flush=True)
        if m["auc"] > mejor:
            mejor = m["auc"]
            torch.save({"pesos": red.state_dict(), "clases": clases, "tam": tam, "tarea": a.tarea,
                        "umbral": m["umbral"]}, salida)

    guardado = torch.load(salida, map_location=disp, weights_only=False)
    red.load_state_dict(guardado["pesos"])
    final = metricas(*predecir(red, c_prueba, disp), umbral=guardado["umbral"])
    guardado.update({"metricas_prueba": final, "historial": historial,
                     "n": {"entrenamiento": len(entr), "validacion": len(val), "prueba": len(prueba)}})
    torch.save(guardado, salida)
    with open(os.path.join(DIR_RESULTADOS, f"entrenamiento_{a.tarea}.json"), "w", encoding="utf-8") as f:
        json.dump({"prueba": final, "historial": historial}, f, indent=2, ensure_ascii=False)

    print(f"\n----- Resultados en PRUEBA ({a.tarea}) -----")
    print(f"Umbral (elegido en validación): {guardado['umbral']:.2f}")
    print(f"AUC={final['auc']:.3f}  Sensibilidad={final['sensibilidad']:.1%}  "
          f"Especificidad={final['especificidad']:.1%}  Exactitud={final['exactitud']:.1%}")
    print("Matriz:", final["matriz"])
    print("Modelo guardado en", salida)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {IA}/entrenar_hueso.py
"""Entrena el clasificador que reconoce qué hueso(s) aparecen en la radiografía.
Sirve para proponer el modelo 3D; el médico siempre confirma.

    python entrenar_hueso.py
"""
import argparse, json, os, random, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

from nucleo import (DIR_MODELOS, DIR_RESULTADOS, MODELO_HUESO, RUTA_DIVISION, asegurar_cache,
                    crear_modelo, dispositivo, transformacion)

MIN_EJEMPLOS = 20  # un hueso con menos imágenes de entrenamiento que esto no se aprende


class Radiografias(Dataset):
    def __init__(self, rutas, etiquetas, entrenamiento):
        self.rutas, self.y = list(rutas), np.asarray(etiquetas, dtype=np.float32)
        self.t = transformacion(entrenamiento)

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, i):
        return self.t(Image.open(self.rutas[i]).convert("RGB")), torch.from_numpy(self.y[i])


def predecir(red, cargador, disp):
    red.eval(); probs, reales = [], []
    with torch.no_grad():
        for x, y in cargador:
            probs.append(torch.sigmoid(red(x.to(disp))).cpu().numpy()); reales.append(y.numpy())
    return np.concatenate(reales), np.concatenate(probs)


def metricas(reales, probs):
    aucs = [roc_auc_score(reales[:, j], probs[:, j]) for j in range(reales.shape[1])
            if 0 < reales[:, j].sum() < len(reales)]
    acierto = float(np.mean(reales[np.arange(len(reales)), probs.argmax(1)] == 1))
    return {"auc_promedio": float(np.mean(aucs)) if aucs else float("nan"), "acierto_hueso_principal": acierto}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epocas", type=int, default=0)
    ap.add_argument("--tam", type=int, default=0)
    ap.add_argument("--lote", type=int, default=32)
    ap.add_argument("--sin-preentrenar", action="store_true")
    a = ap.parse_args()

    disp = dispositivo(); gpu = disp == "cuda"
    epocas = a.epocas or (12 if gpu else 5)
    tam = a.tam or (384 if gpu else 224)
    random.seed(0); np.random.seed(0); torch.manual_seed(0)
    print(f"=== Clasificador de hueso | dispositivo={disp} | épocas={epocas} | imagen={tam}px ===")

    df = pd.read_csv(RUTA_DIVISION)
    cols = [c for c in df.columns if c.startswith("hueso_")]
    entr0 = df[df.conjunto == "entrenamiento"]
    cols = [c for c in cols if entr0[c].sum() >= MIN_EJEMPLOS]
    if len(cols) < 2:
        print("AVISO: el dataset no trae suficientes etiquetas de hueso; se omite este modelo.")
        return
    df = df[df[cols].sum(axis=1) > 0].reset_index(drop=True)
    clases = [c.replace("hueso_", "") for c in cols]
    print("Huesos:", {k: int(df[c].sum()) for k, c in zip(clases, cols)})

    df["cache"] = asegurar_cache(df["ruta"], tam)
    entr, val, prueba = (df[df.conjunto == c] for c in ("entrenamiento", "validacion", "prueba"))
    mk = lambda d, e: DataLoader(Radiografias(d.cache, d[cols].values, e), batch_size=a.lote, shuffle=e, num_workers=0)
    c_entr, c_val, c_prueba = mk(entr, True), mk(val, False), mk(prueba, False)

    red = crear_modelo(len(cols), preentrenado=not a.sin_preentrenar).to(disp)
    pos = entr[cols].sum().values.astype(np.float32)
    peso = torch.tensor(np.clip((len(entr) - pos) / np.maximum(pos, 1), 1, 20), device=disp)
    perdida = nn.BCEWithLogitsLoss(pos_weight=peso)
    opt = torch.optim.AdamW(red.parameters(), lr=3e-4, weight_decay=1e-4)
    plan = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=epocas * len(c_entr))

    os.makedirs(DIR_MODELOS, exist_ok=True); os.makedirs(DIR_RESULTADOS, exist_ok=True)
    mejor, historial = -1, []
    for ep in range(1, epocas + 1):
        red.train(); t0 = time.time(); suma = 0
        for x, y in c_entr:
            x, y = x.to(disp), y.to(disp)
            opt.zero_grad()
            l = perdida(red(x), y)
            l.backward(); opt.step(); plan.step(); suma += l.item()
        m = metricas(*predecir(red, c_val, disp))
        historial.append({"epoca": ep, "perdida": suma / len(c_entr), **m})
        print(f"Época {ep}/{epocas}  pérdida={suma / len(c_entr):.3f}  AUC val={m['auc_promedio']:.3f}  "
              f"acierto={m['acierto_hueso_principal']:.1%}  ({time.time() - t0:.0f}s)", flush=True)
        if m["acierto_hueso_principal"] > mejor:
            mejor = m["acierto_hueso_principal"]
            torch.save({"pesos": red.state_dict(), "clases": clases, "tam": tam, "tarea": "hueso"}, MODELO_HUESO)

    guardado = torch.load(MODELO_HUESO, map_location=disp, weights_only=False)
    red.load_state_dict(guardado["pesos"])
    final = metricas(*predecir(red, c_prueba, disp))
    guardado.update({"metricas_prueba": final, "historial": historial})
    torch.save(guardado, MODELO_HUESO)
    with open(os.path.join(DIR_RESULTADOS, "entrenamiento_hueso.json"), "w", encoding="utf-8") as f:
        json.dump({"prueba": final, "clases": clases, "historial": historial}, f, indent=2, ensure_ascii=False)
    print(f"\n----- Resultados en PRUEBA (hueso) -----\nAUC promedio={final['auc_promedio']:.3f}  "
          f"Acierto del hueso principal={final['acierto_hueso_principal']:.1%}")
    print("Modelo guardado en", MODELO_HUESO)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {IA}/entrenar_aspecto.py
"""Entrena el clasificador del ASPECTO radiológico de la lesión, que decide cómo se dibuja en el 3D:
    crecimiento (sobresale del hueso), litica (hueco) o mixta (destruye hueso y forma masa).
Las etiquetas salen del tipo de tumor de BTXRD (ver ASPECTOS en nucleo.py). El médico siempre confirma.

    python entrenar_aspecto.py
"""
import argparse, json, os, random, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from torch.utils.data import DataLoader, Dataset

from nucleo import (ASPECTOS, DIR_MODELOS, DIR_RESULTADOS, MODELO_ASPECTO, RUTA_DIVISION, asegurar_cache,
                    crear_modelo, dispositivo, transformacion)

MIN_EJEMPLOS = 15


class Radiografias(Dataset):
    def __init__(self, rutas, etiquetas, entrenamiento):
        self.rutas, self.y = list(rutas), np.asarray(etiquetas, dtype=np.int64)
        self.t = transformacion(entrenamiento)

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, i):
        return self.t(Image.open(self.rutas[i]).convert("RGB")), int(self.y[i])


def predecir(red, cargador, disp):
    red.eval(); probs, reales = [], []
    with torch.no_grad():
        for x, y in cargador:
            probs.append(torch.softmax(red(x.to(disp)), 1).cpu().numpy()); reales.append(y.numpy())
    return np.concatenate(reales), np.concatenate(probs)


def metricas(reales, probs, clases):
    pred = probs.argmax(1)
    mc = confusion_matrix(reales, pred, labels=list(range(len(clases))))
    return {"exactitud": float((pred == reales).mean()), "exactitud_balanceada": float(balanced_accuracy_score(reales, pred)),
            "sensibilidad_por_clase": {c: float(mc[i, i] / max(mc[i].sum(), 1)) for i, c in enumerate(clases)},
            "matriz": mc.tolist(), "orden": clases}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epocas", type=int, default=0)
    ap.add_argument("--tam", type=int, default=0)
    ap.add_argument("--lote", type=int, default=32)
    ap.add_argument("--sin-preentrenar", action="store_true")
    a = ap.parse_args()

    disp = dispositivo(); gpu = disp == "cuda"
    epocas = a.epocas or (20 if gpu else 6)
    tam = a.tam or (384 if gpu else 224)
    random.seed(0); np.random.seed(0); torch.manual_seed(0)
    print(f"=== Aspecto de la lesión | dispositivo={disp} | épocas={epocas} | imagen={tam}px ===")

    df = pd.read_csv(RUTA_DIVISION)
    if "aspecto" not in df.columns:
        print("AVISO: vuelve a ejecutar preparar_datos.py (falta la columna 'aspecto'); se omite este modelo."); return
    df = df[df["aspecto"].fillna("").isin(ASPECTOS)].reset_index(drop=True)
    entr0 = df[df.conjunto == "entrenamiento"]
    clases = [c for c in ASPECTOS if (entr0["aspecto"] == c).sum() >= MIN_EJEMPLOS]
    if len(clases) < 2:
        print("AVISO: no hay suficientes lesiones con tipo conocido; se omite este modelo."); return
    df = df[df["aspecto"].isin(clases)].reset_index(drop=True)
    df["y"] = df["aspecto"].map({c: i for i, c in enumerate(clases)})
    print("Imágenes por aspecto:", df.groupby(["aspecto", "conjunto"]).size().unstack(fill_value=0).to_dict("index"))

    df["cache"] = asegurar_cache(df["ruta"], tam)
    entr, val, prueba = (df[df.conjunto == c] for c in ("entrenamiento", "validacion", "prueba"))
    mk = lambda d, e: DataLoader(Radiografias(d.cache, d.y, e), batch_size=a.lote, shuffle=e, num_workers=0)
    c_entr, c_val, c_prueba = mk(entr, True), mk(val, False), mk(prueba, False)

    red = crear_modelo(len(clases), preentrenado=not a.sin_preentrenar).to(disp)
    cuenta = np.bincount(entr.y, minlength=len(clases)).astype(np.float32)
    peso = torch.tensor(cuenta.sum() / (len(clases) * np.maximum(cuenta, 1)), device=disp)   # clases raras pesan más
    perdida = nn.CrossEntropyLoss(weight=peso, label_smoothing=0.05)
    opt = torch.optim.AdamW(red.parameters(), lr=3e-4, weight_decay=1e-4)
    plan = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=epocas * len(c_entr))

    os.makedirs(DIR_MODELOS, exist_ok=True); os.makedirs(DIR_RESULTADOS, exist_ok=True)
    mejor, historial = -1, []
    for ep in range(1, epocas + 1):
        red.train(); t0 = time.time(); suma = 0
        for x, y in c_entr:
            x, y = x.to(disp), y.to(disp)
            opt.zero_grad(); l = perdida(red(x), y); l.backward(); opt.step(); plan.step(); suma += l.item()
        m = metricas(*predecir(red, c_val, disp), clases)
        historial.append({"epoca": ep, "perdida": suma / len(c_entr), "exactitud_balanceada": m["exactitud_balanceada"]})
        print(f"Época {ep}/{epocas}  pérdida={suma / len(c_entr):.3f}  exactitud balanceada val={m['exactitud_balanceada']:.1%}  "
              f"({time.time() - t0:.0f}s)", flush=True)
        if m["exactitud_balanceada"] > mejor:
            mejor = m["exactitud_balanceada"]
            torch.save({"pesos": red.state_dict(), "clases": clases, "tam": tam, "tarea": "aspecto"}, MODELO_ASPECTO)

    guardado = torch.load(MODELO_ASPECTO, map_location=disp, weights_only=False)
    red.load_state_dict(guardado["pesos"])
    final = metricas(*predecir(red, c_prueba, disp), clases)
    guardado.update({"metricas_prueba": final, "historial": historial}); torch.save(guardado, MODELO_ASPECTO)
    with open(os.path.join(DIR_RESULTADOS, "entrenamiento_aspecto.json"), "w", encoding="utf-8") as f:
        json.dump({"prueba": final, "clases": clases, "historial": historial}, f, indent=2, ensure_ascii=False)
    print(f"\n----- Resultados en PRUEBA (aspecto) -----\nExactitud={final['exactitud']:.1%}  "
          f"Exactitud balanceada={final['exactitud_balanceada']:.1%}  Por clase={final['sensibilidad_por_clase']}")
    print("Modelo guardado en", MODELO_ASPECTO)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {IA}/entrenar_detector.py
"""Paso 4: entrena el detector YOLO que dibuja el cuadro sobre la lesión.

    python entrenar_detector.py
"""
import argparse, json, os, shutil

import pandas as pd
from PIL import Image

from nucleo import (DIR_MODELOS, DIR_PREPARADOS, DIR_RESULTADOS, MODELO_DETECTOR, RUTA_CONFIG,
                    RUTA_DIVISION, dispositivo, leer_json)

DIR_YOLO = os.path.join(DIR_PREPARADOS, "yolo")
PARTES = {"entrenamiento": "train", "validacion": "val", "prueba": "test"}


def cajas_de_anotacion(ruta_json):
    """Lee una anotación tipo LabelMe y devuelve los cuadros (x1, y1, x2, y2) en píxeles.
    Si hay rectángulos se usan esos; si solo hay contornos, se usa el cuadro que los encierra."""
    if not ruta_json or not os.path.exists(ruta_json):
        return []
    with open(ruta_json, encoding="utf-8") as f:
        formas = json.load(f).get("shapes", [])
    rect, poli = [], []
    for s in formas:
        pts = s.get("points") or []
        if len(pts) < 2:
            continue
        xs, ys = [p[0] for p in pts], [p[1] for p in pts]
        caja = (min(xs), min(ys), max(xs), max(ys))
        (rect if s.get("shape_type") == "rectangle" or len(pts) == 2 else poli).append(caja)
    return rect or poli


def cajas_yolo(ruta_txt, w, h):
    """Lee recuadros en formato YOLO (fracciones) y los devuelve en píxeles."""
    cajas = []
    if ruta_txt and isinstance(ruta_txt, str) and os.path.exists(ruta_txt):
        for linea in open(ruta_txt, encoding="utf-8"):
            p = linea.split()
            if len(p) == 5:
                cx, cy, bw, bh = (float(v) for v in p[1:])
                cajas.append(((cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h))
    return cajas


def construir_dataset(lado_max=640):
    df = pd.read_csv(RUTA_DIVISION)
    dir_anot = (leer_json(RUTA_CONFIG) or {}).get("dir_anot")
    if not dir_anot:
        raise SystemExit("El dataset no trae la carpeta Annotations; no se puede entrenar el detector.")
    shutil.rmtree(DIR_YOLO, ignore_errors=True)
    con_lesion = df[df["clase"] != "normal"]
    # también algunas radiografías normales como "fondo", para reducir falsas alarmas
    normales = df[df["clase"] == "normal"]
    normales = normales.sample(min(len(normales), max(1, len(con_lesion) // 3)), random_state=0)
    usadas = {p: 0 for p in PARTES.values()}
    for _, f in pd.concat([con_lesion, normales]).iterrows():
        parte = PARTES[f["conjunto"]]
        base = os.path.splitext(f["imagen"])[0]
        img = Image.open(f["ruta"]).convert("RGB")
        w, h = img.size
        cajas = []
        if f["clase"] != "normal":
            if f.get("origen") == "clinica":          # caso corregido por el médico en Orthosync
                cajas = cajas_yolo(f.get("etiquetas_yolo"), w, h)
            else:
                cajas = cajas_de_anotacion(os.path.join(dir_anot, base + ".json"))
            if not cajas:
                continue
        d_img = os.path.join(DIR_YOLO, "images", parte); d_lab = os.path.join(DIR_YOLO, "labels", parte)
        os.makedirs(d_img, exist_ok=True); os.makedirs(d_lab, exist_ok=True)
        if max(w, h) > lado_max:
            e = lado_max / max(w, h)
            img = img.resize((round(w * e), round(h * e)), Image.BILINEAR)
        img.save(os.path.join(d_img, base + ".jpg"), quality=92)
        with open(os.path.join(d_lab, base + ".txt"), "w") as t:
            for x1, y1, x2, y2 in cajas:
                x1, x2 = max(0, min(x1, w)), max(0, min(x2, w)); y1, y2 = max(0, min(y1, h)), max(0, min(y2, h))
                if x2 - x1 < 2 or y2 - y1 < 2:
                    continue
                t.write(f"0 {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}\n")
        usadas[parte] += 1
    for p in PARTES.values():
        os.makedirs(os.path.join(DIR_YOLO, "images", p), exist_ok=True)
    ruta_yaml = os.path.join(DIR_YOLO, "datos.yaml")
    with open(ruta_yaml, "w", encoding="utf-8") as f:
        f.write(f"path: {DIR_YOLO.replace(os.sep, '/')}\ntrain: images/train\nval: images/val\ntest: images/test\n"
                "names:\n  0: lesion\n")
    print("Imágenes para el detector:", usadas)
    if usadas["train"] == 0 or usadas["val"] == 0:
        raise SystemExit("No se encontraron anotaciones de cuadros; no se puede entrenar el detector.")
    return ruta_yaml


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epocas", type=int, default=0)
    ap.add_argument("--tam", type=int, default=0)
    ap.add_argument("--lote", type=int, default=16)
    ap.add_argument("--base", default="", help="Modelo YOLO de partida (por defecto: yolo11s con GPU, yolo11n sin GPU)")
    a = ap.parse_args()

    from ultralytics import YOLO
    disp = dispositivo(); gpu = disp == "cuda"
    # Con GPU: modelo más grande, imágenes más grandes y más épocas (las lesiones pequeñas se ven mejor)
    epocas = a.epocas or (150 if gpu else 30)
    tam = a.tam or (1024 if gpu else 512)
    base = a.base or ("yolo11s.pt" if gpu else "yolo11n.pt")
    lote = a.lote if a.lote != 16 or not gpu else 8
    print(f"=== Detector YOLO ({base}) | dispositivo={disp} | épocas={epocas} | imagen={tam}px ===")

    ruta_yaml = construir_dataset(lado_max=tam)
    modelo = YOLO(base)
    modelo.train(data=ruta_yaml, epochs=epocas, imgsz=tam, batch=lote, device=0 if gpu else "cpu",
                 project=DIR_RESULTADOS, name="detector", exist_ok=True, patience=30, cos_lr=True,
                 workers=0 if os.name == "nt" else 2, seed=0, fliplr=0.5, mosaic=0.5, verbose=False)
    mejor = str(modelo.trainer.best)
    os.makedirs(DIR_MODELOS, exist_ok=True)
    shutil.copy(mejor, MODELO_DETECTOR)

    r = YOLO(MODELO_DETECTOR).val(data=ruta_yaml, split="test", imgsz=tam, device=0 if gpu else "cpu",
                                  project=DIR_RESULTADOS, name="detector_prueba", exist_ok=True,
                                  workers=0, verbose=False)
    m = {"precision": float(r.box.mp), "sensibilidad": float(r.box.mr),
         "mAP50": float(r.box.map50), "mAP50_95": float(r.box.map), "tam": tam, "base": base}
    with open(os.path.join(DIR_RESULTADOS, "metricas_detector.json"), "w", encoding="utf-8") as f:
        json.dump(m, f, indent=2)
    print("\n----- Resultados del detector en PRUEBA -----")
    print(f"mAP50={m['mAP50']:.3f}  mAP50-95={m['mAP50_95']:.3f}  "
          f"Precisión={m['precision']:.1%}  Sensibilidad={m['sensibilidad']:.1%}")
    print("Modelo guardado en", MODELO_DETECTOR)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {IA}/evaluar.py
"""Paso 5: evalúa el sistema completo en el conjunto de PRUEBA y genera
las métricas y figuras para la tesis (carpeta resultados/).

    python evaluar.py
"""
import json, os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

from nucleo import DIR_RESULTADOS, RUTA_DIVISION, Sistema, leer_imagen, leer_json


def intervalo_auc(y, p, n=1000):
    """Intervalo de confianza del 95 % del AUC por remuestreo (bootstrap)."""
    rng = np.random.default_rng(0); vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2:
            vals.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))] if vals else [None, None]


def binaria(y, p, umbral):
    vn, fp, fn, vp = confusion_matrix(y, p >= umbral, labels=[0, 1]).ravel()
    return {
        "n": int(len(y)), "positivos": int(y.sum()), "umbral": float(umbral),
        "auc": float(roc_auc_score(y, p)), "auc_ic95": intervalo_auc(y, p),
        "sensibilidad": float(vp / max(vp + fn, 1)), "especificidad": float(vn / max(vn + fp, 1)),
        "vpp": float(vp / max(vp + fp, 1)), "vpn": float(vn / max(vn + fn, 1)),
        "exactitud": float((vp + vn) / len(y)),
        "f1": float(2 * vp / max(2 * vp + fp + fn, 1)),
        "matriz": {"VN": int(vn), "FP": int(fp), "FN": int(fn), "VP": int(vp)},
    }


def dibujar_matriz(m, etiquetas, titulo, ruta):
    fig, ax = plt.subplots(figsize=(1.6 + 1.3 * len(etiquetas), 1.2 + 1.2 * len(etiquetas)))
    ax.imshow(m, cmap="Blues")
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, int(m[i, j]), ha="center", va="center",
                    color="white" if m[i, j] > m.max() / 2 else "black", fontsize=13)
    ax.set_xticks(range(len(etiquetas)), etiquetas); ax.set_yticks(range(len(etiquetas)), etiquetas)
    ax.set_xlabel("Predicción del sistema"); ax.set_ylabel("Clase real"); ax.set_title(titulo)
    fig.tight_layout(); fig.savefig(ruta, dpi=160); plt.close(fig)


def main():
    os.makedirs(DIR_RESULTADOS, exist_ok=True)
    sistema = Sistema()
    df = pd.read_csv(RUTA_DIVISION)
    df = df[df.conjunto == "prueba"].reset_index(drop=True)
    print(f"Evaluando {len(df)} imágenes de prueba...")
    pt, pm = [], []
    for i, r in enumerate(df["ruta"]):
        a, b = sistema.probabilidades(leer_imagen(r))
        pt.append(a); pm.append(np.nan if b is None else b)
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(df)}", flush=True)
    df["p_tumor"], df["p_maligno"] = pt, pm
    df.drop(columns=["ruta"]).to_csv(os.path.join(DIR_RESULTADOS, "predicciones_prueba.csv"), index=False)

    res, curvas = {}, []
    # Etapa 1: normal vs. lesión
    y = (df["clase"] != "normal").astype(int).values
    res["etapa1_normal_vs_lesion"] = binaria(y, df["p_tumor"].values, sistema.umbral_tumor)
    curvas.append(("Normal vs. lesión", y, df["p_tumor"].values))
    m = res["etapa1_normal_vs_lesion"]["matriz"]
    dibujar_matriz(np.array([[m["VN"], m["FP"]], [m["FN"], m["VP"]]]), ["normal", "lesión"],
                   "Etapa 1: normal vs. lesión", os.path.join(DIR_RESULTADOS, "matriz_etapa1.png"))

    # Etapa 3: benigno vs. maligno (sobre las lesiones reales)
    les = df[df["clase"].isin(["benigno", "maligno"])]
    if sistema.red_mal is not None and les["clase"].nunique() == 2:
        y3 = (les["clase"] == "maligno").astype(int).values
        res["etapa3_benigno_vs_maligno"] = binaria(y3, les["p_maligno"].values, sistema.umbral_malignidad)
        curvas.append(("Benigno vs. maligno", y3, les["p_maligno"].values))
        m = res["etapa3_benigno_vs_maligno"]["matriz"]
        dibujar_matriz(np.array([[m["VN"], m["FP"]], [m["FN"], m["VP"]]]), ["benigno", "maligno"],
                       "Etapa 3: benigno vs. maligno", os.path.join(DIR_RESULTADOS, "matriz_etapa3.png"))

        # Sistema completo: lo que vería el médico (3 clases)
        tres = df[df["clase"].isin(["normal", "benigno", "maligno"])]
        pred = np.where(tres["p_tumor"] < sistema.umbral_tumor, "normal",
                        np.where(tres["p_maligno"] >= sistema.umbral_malignidad, "maligno", "benigno"))
        orden = ["normal", "benigno", "maligno"]
        mc = confusion_matrix(tres["clase"], pred, labels=orden)
        res["sistema_completo_3_clases"] = {
            "exactitud": float(np.trace(mc) / mc.sum()),
            "sensibilidad_por_clase": {c: float(mc[i, i] / max(mc[i].sum(), 1)) for i, c in enumerate(orden)},
            "matriz": mc.tolist(), "orden": orden,
        }
        dibujar_matriz(mc, orden, "Sistema completo", os.path.join(DIR_RESULTADOS, "matriz_sistema.png"))

    det = leer_json(os.path.join(DIR_RESULTADOS, "metricas_detector.json"))
    if det:
        res["etapa2_detector"] = det

    fig, ax = plt.subplots(figsize=(5, 4.5))
    for nombre, yy, pp in curvas:
        fpr, tpr, _ = roc_curve(yy, pp)
        ax.plot(fpr, tpr, lw=2, label=f"{nombre} (AUC = {roc_auc_score(yy, pp):.3f})")
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.set_xlabel("1 - Especificidad"); ax.set_ylabel("Sensibilidad"); ax.set_title("Curvas ROC (conjunto de prueba)")
    ax.legend(loc="lower right"); fig.tight_layout()
    fig.savefig(os.path.join(DIR_RESULTADOS, "curvas_roc.png"), dpi=160); plt.close(fig)

    # puntos de las curvas ROC (reducidos) para dibujarlas en la página de validación
    res["curvas_roc"] = {}
    for nombre, yy, pp in curvas:
        fpr, tpr, _ = roc_curve(yy, pp)
        idx = np.unique(np.linspace(0, len(fpr) - 1, min(len(fpr), 60)).astype(int))
        res["curvas_roc"][nombre] = [[round(float(fpr[i]), 4), round(float(tpr[i]), 4)] for i in idx]

    with open(os.path.join(DIR_RESULTADOS, "metricas.json"), "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2, ensure_ascii=False)

    lineas = ["RESULTADOS EN EL CONJUNTO DE PRUEBA", "=" * 36]
    for clave, titulo in (("etapa1_normal_vs_lesion", "Etapa 1: normal vs. lesión"),
                          ("etapa3_benigno_vs_maligno", "Etapa 3: benigno vs. maligno")):
        if clave in res:
            m = res[clave]
            lineas += ["", titulo, f"  Imágenes: {m['n']} (positivos: {m['positivos']})",
                       f"  AUC: {m['auc']:.3f}  (IC 95 %: {m['auc_ic95'][0]:.3f} - {m['auc_ic95'][1]:.3f})",
                       f"  Sensibilidad: {m['sensibilidad']:.1%}   Especificidad: {m['especificidad']:.1%}",
                       f"  VPP: {m['vpp']:.1%}   VPN: {m['vpn']:.1%}   F1: {m['f1']:.3f}   Exactitud: {m['exactitud']:.1%}",
                       f"  Umbral: {m['umbral']:.2f}   Matriz: {m['matriz']}"]
    if det:
        lineas += ["", "Etapa 2: detector (YOLO)",
                   f"  mAP50: {det['mAP50']:.3f}   mAP50-95: {det['mAP50_95']:.3f}",
                   f"  Precisión: {det['precision']:.1%}   Sensibilidad: {det['sensibilidad']:.1%}"]
    if "sistema_completo_3_clases" in res:
        s = res["sistema_completo_3_clases"]
        lineas += ["", "Sistema completo (normal / benigno / maligno)", f"  Exactitud: {s['exactitud']:.1%}",
                   "  Sensibilidad por clase: " + ", ".join(f"{k} {v:.1%}" for k, v in s["sensibilidad_por_clase"].items())]
    texto = "\n".join(lineas)
    with open(os.path.join(DIR_RESULTADOS, "informe_resultados.txt"), "w", encoding="utf-8") as f:
        f.write(texto + "\n")
    print("\n" + texto)
    print(f"\nFiguras y métricas guardadas en {DIR_RESULTADOS}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {IA}/exportar_onnx.py
"""Convierte los modelos entrenados a ONNX, el formato ligero que usa el servidor web
(no necesita PyTorch instalado). Deja todo en modelos/servidor/.

    python exportar_onnx.py
"""
import json, os, shutil, warnings

import numpy as np
import torch
import torch.nn as nn

from nucleo import (DIR_MODELOS, DIR_RESULTADOS, MODELO_ASPECTO, MODELO_DETECTOR, MODELO_HUESO, MODELO_MALIGNIDAD,
                    MODELO_TUMOR, cargar_clasificador, leer_json)

DIR_SERVIDOR = os.path.join(DIR_MODELOS, "servidor")


class ConMapa(nn.Module):
    """Devuelve las probabilidades y, además, el mapa de activación de la clase positiva
    (equivalente a Grad-CAM en una ResNet), para dibujar el mapa de calor sin PyTorch."""

    def __init__(self, red):
        super().__init__(); self.red = red

    def forward(self, x):
        r = self.red
        f = r.layer4(r.layer3(r.layer2(r.layer1(r.maxpool(r.relu(r.bn1(r.conv1(x))))))))
        logits = r.fc(torch.flatten(r.avgpool(f), 1))
        mapa = torch.relu((f * r.fc.weight[1].view(1, -1, 1, 1)).sum(1))
        return torch.softmax(logits, 1), mapa


class Multiclase(nn.Module):
    def __init__(self, red):
        super().__init__(); self.red = red

    def forward(self, x):
        return torch.softmax(self.red(x), 1)


class Multietiqueta(nn.Module):
    def __init__(self, red):
        super().__init__(); self.red = red

    def forward(self, x):
        return torch.sigmoid(self.red(x))


def exportar(modulo, tam, destino, salidas):
    modulo.eval()
    x = torch.randn(1, 3, tam, tam)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            torch.onnx.export(modulo, (x,), destino, input_names=["imagen"], output_names=salidas,
                              opset_version=17, dynamo=False)
        except Exception:  # versiones de PyTorch con otro exportador por defecto
            torch.onnx.export(modulo, (x,), destino, input_names=["imagen"], output_names=salidas)
    import onnxruntime as ort
    s = ort.InferenceSession(destino, providers=["CPUExecutionProvider"])
    con_onnx = s.run(None, {"imagen": x.numpy()})
    with torch.no_grad():
        con_torch = modulo(x)
    con_torch = con_torch if isinstance(con_torch, tuple) else (con_torch,)
    dif = max(float(np.abs(a - b.numpy()).max()) for a, b in zip(con_onnx, con_torch))
    print(f"  {os.path.basename(destino)}: {os.path.getsize(destino) / 1e6:.1f} MB, diferencia máxima con PyTorch {dif:.2e}")
    if dif > 1e-3:
        raise SystemExit("La versión ONNX no coincide con el modelo original.")


def main():
    os.makedirs(DIR_SERVIDOR, exist_ok=True)
    info = {"modelos": {}}
    for clave, ruta in (("tumor", MODELO_TUMOR), ("malignidad", MODELO_MALIGNIDAD)):
        if os.path.exists(ruta):
            red, d = cargar_clasificador(ruta)
            exportar(ConMapa(red), d["tam"], os.path.join(DIR_SERVIDOR, f"{clave}.onnx"), ["probabilidades", "mapa"])
            info["modelos"][clave] = {"archivo": f"{clave}.onnx", "tam": d["tam"], "clases": d["clases"],
                                      "umbral": d.get("umbral", 0.5), "metricas_prueba": d.get("metricas_prueba")}
    if os.path.exists(MODELO_HUESO):
        red, d = cargar_clasificador(MODELO_HUESO)
        exportar(Multietiqueta(red), d["tam"], os.path.join(DIR_SERVIDOR, "hueso.onnx"), ["probabilidades"])
        info["modelos"]["hueso"] = {"archivo": "hueso.onnx", "tam": d["tam"], "clases": d["clases"],
                                    "metricas_prueba": d.get("metricas_prueba")}
    if os.path.exists(MODELO_ASPECTO):
        red, d = cargar_clasificador(MODELO_ASPECTO)
        exportar(Multiclase(red), d["tam"], os.path.join(DIR_SERVIDOR, "aspecto.onnx"), ["probabilidades"])
        info["modelos"]["aspecto"] = {"archivo": "aspecto.onnx", "tam": d["tam"], "clases": d["clases"],
                                      "metricas_prueba": d.get("metricas_prueba")}
    if os.path.exists(MODELO_DETECTOR):
        from ultralytics import YOLO
        tam = int((leer_json(os.path.join(DIR_RESULTADOS, "metricas_detector.json")) or {}).get("tam", 640))
        salida = YOLO(MODELO_DETECTOR).export(format="onnx", imgsz=tam, opset=17, simplify=False, verbose=False)
        shutil.move(str(salida), os.path.join(DIR_SERVIDOR, "detector.onnx"))
        print(f"  detector.onnx: {os.path.getsize(os.path.join(DIR_SERVIDOR, 'detector.onnx')) / 1e6:.1f} MB")
        info["modelos"]["detector"] = {"archivo": "detector.onnx", "tam": tam, "clases": ["lesion"],
                                       "metricas_prueba": leer_json(os.path.join(DIR_RESULTADOS, "metricas_detector.json"))}
    info["preprocesamiento"] = {"clahe": {"clip": 2.0, "rejilla": 8}, "media": [0.485, 0.456, 0.406],
                                "desviacion": [0.229, 0.224, 0.225],
                                "nota": "escala de grises -> CLAHE -> redimensionar con relleno negro -> RGB -> normalizar"}
    info["evaluacion"] = leer_json(os.path.join(DIR_RESULTADOS, "metricas.json"))
    with open(os.path.join(DIR_SERVIDOR, "info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)
    print("Modelos para el servidor en", DIR_SERVIDOR, "->", sorted(os.listdir(DIR_SERVIDOR)))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {IA}/entrenar_todo.py
"""Entrena y evalúa el sistema completo con un solo comando.

    python entrenar_todo.py --datos "C:\\tesis\\BTXRD"            (completo)
    python entrenar_todo.py --datos "C:\\tesis\\BTXRD" --rapido   (prueba de pocos minutos)
"""
import argparse, os, subprocess, sys, time

AQUI = os.path.dirname(os.path.abspath(__file__))


def paso(titulo, *args):
    print(f"\n{'#' * 70}\n# {titulo}\n{'#' * 70}", flush=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, *args], cwd=AQUI)
    if r.returncode != 0:
        raise SystemExit(f"\nERROR en el paso: {titulo}. Revisa el mensaje de arriba.")
    print(f"[{titulo}] listo en {(time.time() - t0) / 60:.1f} min", flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--datos", required=True, help="Carpeta donde descomprimiste BTXRD")
    ap.add_argument("--rapido", action="store_true", help="Prueba rápida con pocas imágenes y épocas")
    ap.add_argument("--sin-detector", action="store_true", help="No entrenar YOLO (es el paso más lento)")
    ap.add_argument("--extra", nargs="*", default=[], help="Carpetas con casos corregidos exportados desde Orthosync")
    a = ap.parse_args()

    prep = ["preparar_datos.py", "--datos", a.datos]
    c1 = ["entrenar_clasificador.py", "--tarea", "tumor"]
    c2 = ["entrenar_clasificador.py", "--tarea", "malignidad"]
    det = ["entrenar_detector.py"]
    hueso = ["entrenar_hueso.py"]
    aspecto = ["entrenar_aspecto.py"]
    if a.extra:
        prep += ["--extra", *a.extra]
    if a.rapido:
        prep += ["--max-imagenes", "450"]
        c1 += ["--epocas", "2"]; c2 += ["--epocas", "2"]
        det += ["--epocas", "3", "--tam", "320"]
        hueso += ["--epocas", "2"]; aspecto += ["--epocas", "2"]

    paso("1/7 Preparar y dividir los datos", *prep)
    paso("2/7 Clasificador normal vs. lesión", *c1)
    paso("3/7 Clasificador benigno vs. maligno", *c2)
    paso("4/7 Clasificador de hueso", *hueso)
    paso("4b/7 Aspecto de la lesión (para el 3D)", *aspecto)
    if not a.sin_detector:
        paso("5/7 Detector YOLO (localización)", *det)
    paso("6/7 Evaluación en el conjunto de prueba", "evaluar.py")
    paso("7/7 Exportar los modelos para el servidor (ONNX)", "exportar_onnx.py")
    print("\nTODO LISTO. Los modelos están en la carpeta modelos/ y las métricas en resultados/")


if __name__ == "__main__":
    main()


## 2. Dataset BTXRD
Se descarga solo desde figshare. Si la descarga falla, sube tu `BTXRD.zip` a Kaggle
(*Add Input → Upload → New Dataset*), agrégalo a este cuaderno y vuelve a ejecutar: el cuaderno lo encontrará en `/kaggle/input`.

In [ ]:
import requests, zipfile

def buscar_tabla(raiz):
    for d, _, archivos in os.walk(raiz):
        if any(a.lower() in ('dataset.xlsx', 'dataset.csv') for a in archivos):
            return d
    return None

def buscar_casos(raiz):
    # carpetas con casos corregidos exportados desde Orthosync (contienen casos.csv)
    return sorted({d for d, _, archivos in os.walk(raiz) if 'casos.csv' in archivos}) if os.path.isdir(raiz) else []

DATOS = os.environ.get('ORTHOSYNC_DATOS') or (buscar_tabla('/kaggle/input') if os.path.isdir('/kaggle/input') else None)
CASOS = buscar_casos('/kaggle/input')
for z in [os.path.join(d, a) for d, _, archivos in os.walk('/kaggle/input') for a in archivos
          if a.startswith('orthosync_casos') and a.endswith('.zip')] if os.path.isdir('/kaggle/input') else []:
    destino = os.path.join(BASE, 'casos', os.path.splitext(os.path.basename(z))[0])
    with zipfile.ZipFile(z) as zz:
        zz.extractall(destino)
    CASOS.append(destino)
print('Casos corregidos por el médico:', CASOS or 'ninguno')
if not DATOS:
    destino = os.path.join(BASE, 'BTXRD'); os.makedirs(destino, exist_ok=True)
    DATOS = buscar_tabla(destino)
    if not DATOS:
        cab = {'User-Agent': 'Mozilla/5.0'}
        info = requests.get('https://api.figshare.com/v2/articles/27865398', headers=cab, timeout=60).json()
        for f in info['files']:
            print('Descargando', f['name'], round(f['size'] / 1e6), 'MB')
            ruta = os.path.join(destino, f['name'])
            with requests.get(f['download_url'], headers=cab, stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(ruta, 'wb') as out:
                    for parte in r.iter_content(1 << 20):
                        out.write(parte)
            if ruta.endswith('.zip'):
                with zipfile.ZipFile(ruta) as z:
                    z.extractall(destino)
                os.remove(ruta)
        DATOS = buscar_tabla(destino)
assert DATOS, 'No se encontró el dataset. Súbelo a Kaggle como se indica arriba.'
print('Dataset en:', DATOS)

## 3. Entrenar
`MODO = 'completo'` entrena todo (con GPU T4, calculo entre 3 y 6 horas: el detector ahora es más grande).
`MODO = 'rapido'` es una prueba de pocos minutos para comprobar que todo corre; sus resultados no sirven.

In [ ]:
MODO = os.environ.get('ORTHOSYNC_MODO', 'completo')
orden = [sys.executable, 'entrenar_todo.py', '--datos', DATOS] + (['--rapido'] if MODO == 'rapido' else []) + (['--extra', *CASOS] if CASOS else [])
p = subprocess.Popen(orden, cwd=IA, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
for linea in p.stdout:
    if '━' not in linea and 'it/s' not in linea:   # oculta las barras de progreso de YOLO
        print(linea, end='')
assert p.wait() == 0, 'El entrenamiento terminó con error; revisa el mensaje de arriba.'

## 4. Resultados

In [ ]:
from IPython.display import Image, display
print(open(os.path.join(IA, 'resultados', 'informe_resultados.txt'), encoding='utf-8').read())
for f in ['curvas_roc.png', 'matriz_etapa1.png', 'matriz_etapa3.png', 'matriz_sistema.png']:
    ruta = os.path.join(IA, 'resultados', f)
    if os.path.exists(ruta):
        display(Image(ruta, width=420))

## 5. Empaquetar para descargar
Genera `orthosync_modelos.zip` con los modelos y las métricas, y borra los archivos temporales.

In [ ]:
import shutil
paquete = os.path.join(BASE, '_paquete'); shutil.rmtree(paquete, ignore_errors=True)
shutil.copytree(os.path.join(IA, 'modelos'), os.path.join(paquete, 'modelos'))
shutil.copytree(os.path.join(IA, 'resultados'), os.path.join(paquete, 'resultados'))
shutil.copytree(os.path.join(IA, 'ejemplos_prueba'), os.path.join(paquete, 'ejemplos_prueba'))
shutil.make_archive(os.path.join(BASE, 'orthosync_modelos'), 'zip', paquete)
for sobra in [paquete, os.path.join(IA, 'datos_preparados'), os.path.join(BASE, 'BTXRD')]:
    shutil.rmtree(sobra, ignore_errors=True)
print('Listo:', os.path.join(BASE, 'orthosync_modelos.zip'), round(os.path.getsize(os.path.join(BASE, 'orthosync_modelos.zip')) / 1e6), 'MB')